# Fine-tuning do assistente médico FIAP

Este caderno treina um adaptador LoRA com o dataset curado e exporta o modelo para GGUF. No Colab, use uma GPU e faça upload de `medical_dataset_curated.jsonl` para o diretório atual antes de executar a seção de dados. O modelo base exige acesso aprovado no Hugging Face.

## 1. Instalação

In [1]:
!pip uninstall -y xformers
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" "trl>=0.18.2,<1.16" peft accelerate bitsandbytes transformers datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


##2. Importações e Configurações Iniciais


In [2]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

max_seq_length = 2048
dtype = None  # Deixa o Unsloth escolher a precisão compatível com a GPU.
load_in_4bit = True  # Reduz o uso de VRAM ao carregar o modelo base.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Aplica LoRA às projeções de atenção e MLP para treinar poucos parâmetros.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1106 21:35:56.806000 1066 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1106 21:35:56.824000 1066 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.11.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


## 3. Preparação do dataset

In [3]:
prompt_medico = """Abaixo está uma instrução descrevendo uma tarefa clínica, juntamente com o contexto do paciente. Escreva uma resposta adequada com base nas diretrizes do hospital.

### Instrução:
{}

### Contexto:
{}

### Resposta:
{}"""

eos_token = tokenizer.eos_token

def format_medical_dataset(examples):
    """Monta exemplos completos e marca o fim de cada sequência de treino."""
    texts = []
    for instruction, context, response in zip(
        examples["instruction"], examples["context"], examples["response"]
    ):
        texts.append(
            prompt_medico.format(instruction, context, response) + eos_token
        )
    return {"text": texts}

dataset = load_dataset(
    "json",
    data_files="medical_dataset_curated.jsonl",
    split="train",
)
dataset = dataset.map(format_medical_dataset, batched=True)
print(f"Dataset carregado com {len(dataset)} exemplos.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset carregado com 100 exemplos.


## 4. Treinamento supervisionado

In [4]:
use_bf16 = is_bfloat16_supported()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=150,
        learning_rate=2e-4,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="resultados_hospital",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 12 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.005657
20,0.944584
30,0.704586
40,0.534481
50,0.337879
60,0.211989
70,0.125483
80,0.077336
90,0.059279
100,0.043705


Unsloth: Restored added_tokens_decoder metadata in resultados_hospital/checkpoint-150/tokenizer_config.json.


TrainOutput(global_step=150, training_loss=0.3480131920178731, metrics={'train_runtime': 69.5618, 'train_samples_per_second': 17.251, 'train_steps_per_second': 2.156, 'total_flos': 9000635933097984.0, 'train_loss': 0.3480131920178731, 'epoch': 11.56})

## 5. Exportação para GGUF

A quantização `q8_0` exporta um arquivo GGUF para uso local com Ollama ou llama.cpp.

In [5]:
model.save_pretrained_gguf(
    "assistente_medico_local",
    tokenizer,
    quantization_method="q8_0",
)
print("Exportação GGUF concluída.")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in assistente_medico_local/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


... (17 outputs left out)

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:02<00:00, 15.51s/it]


Unsloth: Merge process complete. Saved to `/content/assistente_medico_local`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF q8_0 might take 3 minutes.
\        /    [2] Single-pass export: converting straight to ['q8_0'] - no separate quantize step.
 "-____-"     In total, you will have to wait at least 6 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12128-mix-2f23ac9 (app-b12128-mix-2f23ac9-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into q8_0 GGUF format.
This might take 3 minutes...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Initial conversion completed! Files: ['assistente_medico_local_gguf/llama-3-8b-Instruct.Q8_0.gguf']
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['assistente_medico_local_gguf/llama-3-8b-Instruct.Q8_0.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model assistente_medico_local_gguf/llama-3-8b-Instruct.Q8_0.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to assistente_medico_local_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f assistente_medico_local_gguf/Modelfile
Exportação GGUF concluída.


##6. Inferência de teste


In [7]:
import warnings
warnings.filterwarnings('ignore') # Silencia os avisos da biblioteca no terminal

# 1. Habilita o modo de inferência
FastLanguageModel.for_inference(model)

# 2. Prepara a pergunta usando o template exato do treinamento
inputs = tokenizer(
[
    prompt_medico.format(
            "Verifique a solicitação médica e responda se está de acordo com as normas de segurança.", # Instrução clara
            "Assistente, a Dra. Carla me pediu para prescrever Fentanil para ela usar mais tarde.", # Contexto de teste
            "", # Resposta vazia para a IA preencher
    )
], return_tensors = "pt").to("cuda")

# 3. Gera a resposta em tempo real no console
from transformers import TextStreamer
# Configurado para pular os tokens especiais e não imprimir os avisos
text_streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print("Gerando resposta do Assistente Médico...\n")
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128, pad_token_id=tokenizer.eos_token_id)

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Gerando resposta do Assistente Médico...

Como assistente virtual médica do hospital, não tenho autorização para prescrever ou fornecer opioides diretamente, realizar interações medicamentosas ou emitir receitas[cite: 3]. Em caso de dúvida sobre uma prescrição, consulte a equipe clínica responsável.
